### 连接到虚拟机的apache spark

In [ ]:
from pyspark.sql import SparkSession
from pyspark import StorageLevel
from time import perf_counter

spark = SparkSession.builder \
    .remote("sc://<vm-hostname>:15002") \
    .getOrCreate()

print("连接成功！Spark 版本:", spark.version)
spark.sql("SHOW DATABASES").show()

spark.conf.set("spark.sql.parquet.outputTimestampType", "TIMESTAMP_MICROS")
spark.conf.set("spark.sql.legacy.parquet.nanosAsLong", "true")

连接成功！Spark 版本: 3.5.7
+---------+
|namespace|
+---------+
|  default|
+---------+



### 连接到虚拟机的Hadoop hdfs

In [ ]:
# 2) HDFS 路径
hdfs_base = "hdfs://<HDFS-url>:9000"
geo_path = f"{hdfs_base}/spark/httplog_analytics/01_geo_dim.parquet"
url_path = f"{hdfs_base}/spark/httplog_analytics/02_url_category_dim.parquet"
log_path = f"{hdfs_base}/spark/httplog_analytics/03_http_logs.parquet"

print("正在读取 HDFS 中的 Parquet 文件...")
geo_df = spark.read.parquet(geo_path)
url_df = spark.read.parquet(url_path)
logs_df = spark.read.parquet(log_path)

print("geo_df 记录数:", geo_df.count())
print("url_df 记录数:", url_df.count())
print("logs_df 记录数:", logs_df.count())

# 3) 查看表结构
print("\n=== geo_dim schema ===")
geo_df.printSchema()
print("\n=== url_category_dim schema ===")
url_df.printSchema()
print("\n=== http_logs schema ===")
logs_df.printSchema()

正在读取 HDFS 中的 Parquet 文件...
geo_df 记录数: 32
url_df 记录数: 800
logs_df 记录数: 600000

=== geo_dim schema ===
root
 |-- country_code: string (nullable = true)
 |-- country_name: string (nullable = true)
 |-- region: string (nullable = true)
 |-- continent: string (nullable = true)
 |-- is_high_risk: boolean (nullable = true)


=== url_category_dim schema ===
root
 |-- path_pattern: string (nullable = true)
 |-- category: string (nullable = true)
 |-- is_api: boolean (nullable = true)
 |-- is_static: boolean (nullable = true)
 |-- priority: long (nullable = true)


=== http_logs schema ===
root
 |-- log_id: string (nullable = true)
 |-- timestamp: long (nullable = true)
 |-- date: string (nullable = true)
 |-- hour: long (nullable = true)
 |-- ip: string (nullable = true)
 |-- method: string (nullable = true)
 |-- url: string (nullable = true)
 |-- path: string (nullable = true)
 |-- status_code: long (nullable = true)
 |-- response_time_ms: long (nullable = true)
 |-- bytes: long (nullable = t

### 几张原始数据表进行并联

In [3]:
# 5) 联表：日志 + geo_dim + url_category_dim
# 5) 联表：使用 broadcast join（对 small dim 表使用 broadcast）
from pyspark.sql.functions import broadcast
log_geo_df = logs_df.join(
    broadcast(geo_df),
    logs_df["country"] == geo_df["country_code"],
    "left"
)

log_geo_url_df = log_geo_df.join(
    broadcast(url_df),
    log_geo_df["path"] == url_df["path_pattern"],
    "left"
)

# 对刚创建的中间表做一次 persist（用于后续多次聚合）
log_geo_url_df = log_geo_url_df.repartition("country")  # 按业务常用维度分区
log_geo_url_df.persist(StorageLevel.MEMORY_AND_DISK)

# 6) 关键字段展示
print("\n=== 采样数据 ===")
log_geo_url_df.select(
    "log_id","timestamp","country","country_name","region","continent","path","category","status_code","response_time_ms"
).show(10, truncate=False)



=== 采样数据 ===
+-----------------+-------------------+-------+------------+--------------+---------+-----------------------+--------+-----------+----------------+
|log_id           |timestamp          |country|country_name|region        |continent|path                   |category|status_code|response_time_ms|
+-----------------+-------------------+-------+------------+--------------+---------+-----------------------+--------+-----------+----------------+
|20240107-00000028|1704649516000000000|UA     |Ukraine     |Eastern Europe|Europe   |/s/coffee              |search  |200        |2878            |
|20240107-00000173|1704604607000000000|UA     |Ukraine     |Eastern Europe|Europe   |/list/electronics      |category|200        |264             |
|20240106-00000245|1704567155000000000|UA     |Ukraine     |Eastern Europe|Europe   |/goods/1780            |product |404        |3056            |
|20240102-00000271|1704213080000000000|UA     |Ukraine     |Eastern Europe|Europe   |/manage/syste

### timestamp格式转换

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql import Window

# 如果已有时间列 ts 可复用，否则根据 timestamp 生成（与 notebook 中 to_ts_expr 保持一致）
if "ts" not in log_geo_url_df.columns:
    max_ts = logs_df.agg(F.max("timestamp")).collect()[0][0]

    def to_ts_expr(col):
        if max_ts > 1e17:
            return F.to_timestamp((F.col(col) / F.lit(1e9)).cast("double"))
        elif max_ts > 1e14:
            return F.to_timestamp((F.col(col) / F.lit(1e6)).cast("double"))
        elif max_ts > 1e11:
            return F.to_timestamp((F.col(col) / F.lit(1e3)).cast("double"))
        else:
            return F.to_timestamp(F.from_unixtime(F.col(col).cast("long")))

    log_ts = log_geo_url_df.withColumn("ts", to_ts_expr("timestamp"))
else:
    log_ts = log_geo_url_df


In [26]:
import os

local_dir = r"G:\Log-Spark-Insight-Analysis-v2\output_further_analytics"
os.makedirs(local_dir, exist_ok=True)

### 网络流量的总体画像

In [29]:
from pyspark.sql import functions as F
from pyspark.sql import Window
# 基本总体画像指标
summary = log_ts.agg(
    F.count("*").alias("total_requests"),
    F.min("ts").alias("min_ts"),
    F.max("ts").alias("max_ts"),
    F.countDistinct("ip").alias("distinct_ips"),
    F.countDistinct("path").alias("distinct_paths"),
    F.countDistinct("country").alias("distinct_countries")
).withColumn(
    "time_span_seconds",
    F.unix_timestamp(F.col("max_ts")) - F.unix_timestamp(F.col("min_ts"))
)

print("\n=== 流量总体画像（摘要） ===")
summary.show(truncate=False)
pd_summary = summary.toPandas()
pd_summary.attrs.clear()
pd_summary.to_parquet(os.path.join(local_dir, "01_network_traffic_overview.parquet"), index=False)

# 请求方法分布（兼容 method 列可能缺失）
if "method" in log_ts.columns:
    method_dist = (
        log_ts.groupBy("method")
        .count()
        .withColumn(
            "pct",
            F.round(
                F.col("count")
                / F.sum("count").over(Window.rowsBetween(Window.unboundedPreceding, Window.unboundedFollowing))
                * 100,
                2,
            ),
        )
        .orderBy(F.desc("count"))
    )
    print("\n=== 请求方法分布 ===")
    method_dist.show(truncate=False)
    pd_method_dist = method_dist.toPandas()
    pd_method_dist.attrs.clear()
    pd_method_dist.to_parquet(os.path.join(local_dir, "02_request_method_dist.parquet"), index=False)
else:
    print("\n=== 请求方法列 `method` 不存在，跳过方法分布 ===")

# 协议/端口分布（兼容 protocol/port 列）
prot_cols = [c for c in ("protocol", "port") if c in log_ts.columns]
if prot_cols:
    prot_group = log_ts.groupBy(*prot_cols).count().orderBy(F.desc("count"))
    print("\n=== 协议/端口 分布 ===")
    prot_group.show(50, truncate=False)
    pd_prot_group = prot_group.toPandas()
    pd_prot_group.attrs.clear()
    pd_prot_group.to_parquet(os.path.join(local_dir, "03_protocol_port_dist.parquet"), index=False)
else:
    print("\n=== 未检测到 `protocol` 或 `port` 列，跳过协议/端口分布 ===")

# User-Agent 粗分类：浏览器 / 爬虫 / 脚本 / 其他（兼容 user_agent 或 ua 列）
ua_col = None
for c in ("user_agent", "ua", "client_agent"):
    if c in log_ts.columns:
        ua_col = c
        break

if ua_col:
    ua_df = log_ts.withColumn("ua_lc", F.lower(F.coalesce(F.col(ua_col), F.lit(""))))
    ua_df = ua_df.withColumn(
        "ua_type",
        F.when(ua_df.ua_lc.rlike("bot|spider|crawler|bingpreview|slurp|duckduckbot|baiduspider|yandex"), "crawler")
        .when(ua_df.ua_lc.rlike("curl|wget|python-requests|httpclient|java|okhttp|libwww-perl|go-http-client"), "script")
        .when(ua_df.ua_lc.rlike("mozilla|chrome|safari|firefox|edge|opera"), "browser")
        .otherwise("other"),
    )
    ua_stats = (
        ua_df.groupBy("ua_type")
        .count()
        .withColumn(
            "pct",
            F.round(
                F.col("count")
                / F.sum("count").over(Window.rowsBetween(Window.unboundedPreceding, Window.unboundedFollowing))
                * 100,
                2,
            ),
        )
        .orderBy(F.desc("count"))
    )
    print("\n=== User-Agent 大类分布（browser/crawler/script/other） ===")
    ua_stats.show(truncate=False)
    pd_ua_stats = ua_stats.toPandas()
    pd_ua_stats.attrs.clear()
    pd_ua_stats.to_parquet(os.path.join(local_dir, "04_ua_type_dist.parquet"), index=False)
else:
    print("\n=== 未检测到 User-Agent 列（user_agent/ua/client_agent），跳过 UA 分类 ===")

# 将主要结果收集为单行 Pandas（便于保存/展示）
pd_summary


=== 流量总体画像（摘要） ===
+--------------+-------------------+-------------------+------------+--------------+------------------+-----------------+
|total_requests|min_ts             |max_ts             |distinct_ips|distinct_paths|distinct_countries|time_span_seconds|
+--------------+-------------------+-------------------+------------+--------------+------------------+-----------------+
|600000        |2024-01-01 08:00:02|2024-01-11 07:59:39|589964      |800           |32                |863977           |
+--------------+-------------------+-------------------+------------+--------------+------------------+-----------------+


=== 请求方法分布 ===


d:\miniconda_envs\junliangvenv_hive1\Lib\site-packages\pyspark\sql\connect\expressions.py:948: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+------+------+-----+
|method|count |pct  |
+------+------+-----+
|GET   |148683|24.78|
|POST  |109286|18.21|
|PUT   |93942 |15.66|
|DELETE|93297 |15.55|
|HEAD  |77899 |12.98|
|PATCH |76893 |12.82|
+------+------+-----+


=== 未检测到 `protocol` 或 `port` 列，跳过协议/端口分布 ===

=== User-Agent 大类分布（browser/crawler/script/other） ===


d:\miniconda_envs\junliangvenv_hive1\Lib\site-packages\pyspark\sql\connect\expressions.py:948: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+-------+------+-----+
|ua_type|count |pct  |
+-------+------+-----+
|browser|342283|57.05|
|script |172037|28.67|
|other  |85680 |14.28|
+-------+------+-----+



,total_requests,min_ts,max_ts,distinct_ips,distinct_paths,distinct_countries,time_span_seconds
0,600000,2024-01-01 08:00:02,2024-01-11 07:59:39,589964,800,32,863977


### 按国家统计URL的访问量

In [30]:
# 7) 统计分析 1：按国家统计访问量（Network Traffic Analysis）
print("\n=== 1. 各国家访问量 Top 10 ===")
country_stats = (
    log_ts.groupBy("country_name", "country")
    .count()
    .orderBy("count", ascending=False)
    .limit(10)
)
country_stats.show(truncate=False)

pd_country = country_stats.toPandas()
pd_country.attrs.clear()   
pd_country.to_parquet(os.path.join(local_dir, "05_country_traffic_stats.parquet"), index=False)


=== 1. 各国家访问量 Top 10 ===
+--------------+-------+------+
|country_name  |country|count |
+--------------+-------+------+
|China         |CN     |106075|
|United States |US     |70835 |
|India         |IN     |52669 |
|Japan         |JP     |35008 |
|France        |FR     |26753 |
|United Kingdom|GB     |26624 |
|Germany       |DE     |26336 |
|Australia     |AU     |17680 |
|Canada        |CA     |17562 |
|Brazil        |BR     |17539 |
+--------------+-------+------+



### URL访问类型的数量统计

In [31]:
# 8) 统计分析 2：按 URL 分类统计访问量
print("\n=== 2. 各 URL 分类访问量 Top 10 ===")
category_stats = (
    log_ts.groupBy("category")
    .count()
    .orderBy("count", ascending=False)
    .limit(10)
)
category_stats.show(truncate=False)
pd_category = category_stats.toPandas()
pd_category.attrs.clear()
pd_category.to_parquet(os.path.join(local_dir, "06_url_category_traffic_stats.parquet"), index=False)


=== 2. 各 URL 分类访问量 Top 10 ===
+--------+-----+
|category|count|
+--------+-----+
|home    |97874|
|product |93063|
|category|75873|
|api     |66395|
|static  |54075|
|user    |43283|
|search  |38588|
|login   |38130|
|other   |37810|
|checkout|21801|
+--------+-----+



### URL状态码分布

In [ ]:
# 9) 统计分析 3：HTTP 状态码分布
print("\n=== 3. HTTP 状态码分布 ===")
status_stats = (
    log_ts.groupBy("status_code")
    .count()
    .orderBy("count", ascending=False)
)
status_stats.show(20, truncate=False)
pd_status_stats = status_stats.toPandas()
pd_status_stats.attrs.clear()
pd_status_stats.to_parquet(os.path.join(local_dir, "07_http_status_code_dist.parquet"), index=False)

# 计算 2xx / 3xx 的数量与占比
tot = log_ts.count()
status_2xx_3xx = (
    log_ts.agg(
        F.sum(F.when((F.col("status_code") >= 200) & (F.col("status_code") < 300), 1).otherwise(0)).alias("cnt_2xx"),
        F.sum(F.when((F.col("status_code") >= 300) & (F.col("status_code") < 400), 1).otherwise(0)).alias("cnt_3xx"),
    )
    .withColumn("total_requests", F.lit(tot))
    .withColumn("pct_2xx", F.round(F.col("cnt_2xx") / F.col("total_requests") * 100, 2))
    .withColumn("pct_3xx", F.round(F.col("cnt_3xx") / F.col("total_requests") * 100, 2))
)
print("\n=== 2xx / 3xx 汇总 ===")
status_2xx_3xx.show(truncate=False)
pd_status_2xx_3xx = status_2xx_3xx.toPandas()
pd_status_2xx_3xx.attrs.clear()
pd_status_2xx_3xx.to_parquet(os.path.join(local_dir, "08_http_status_2xx_3xx_summary.parquet"), index=False)

# 分别展示 2xx 和 3xx 的详细 top status_code
print("\n=== 2xx 细分 Top ===")
log_ts.filter((F.col("status_code") >= 200) & (F.col("status_code") < 300)).groupBy("status_code").count().orderBy(F.desc("count")).show(20, truncate=False)

print("\n=== 3xx 细分 Top ===")
log_ts.filter((F.col("status_code") >= 300) & (F.col("status_code") < 400)).groupBy("status_code").count().orderBy(F.desc("count")).show(20, truncate=False)



=== 3. HTTP 状态码分布 ===
+-----------+------+
|status_code|count |
+-----------+------+
|200        |416614|
|500        |46908 |
|404        |46694 |
|302        |42020 |
|301        |41876 |
|503        |5141  |
|403        |696   |
|401        |51    |
+-----------+------+


=== 2xx / 3xx 汇总 ===
+-------+-------+--------------+-------+-------+
|cnt_2xx|cnt_3xx|total_requests|pct_2xx|pct_3xx|
+-------+-------+--------------+-------+-------+
|416614 |83896  |600000        |69.44  |13.98  |
+-------+-------+--------------+-------+-------+


=== 2xx 细分 Top ===
+-----------+------+
|status_code|count |
+-----------+------+
|200        |416614|
+-----------+------+


=== 3xx 细分 Top ===
+-----------+-----+
|status_code|count|
+-----------+-----+
|302        |42020|
|301        |41876|
+-----------+-----+



### 统计HTTP的请求量

In [33]:
from pyspark.sql import functions as F

# 10) 按小时统计请求量（输出包含 date）
print("\n=== 4. 按小时统计请求量（含 date） ===")
hourly_stats = (
    log_ts
    .withColumn("date", F.to_date("ts"))
    .withColumn("hour", F.hour("ts"))
    .groupBy("date", "hour")
    .count()
    .orderBy("date", "hour")
)
hourly_stats.show(200, truncate=False)
pd_hourly_stats = hourly_stats.toPandas()
pd_hourly_stats.attrs.clear()
pd_hourly_stats.to_parquet(os.path.join(local_dir, "09_hourly_traffic_stats.parquet"), index=False)

# 10b) 按天统计请求量（保留）
print("\n=== 4b. 按天统计请求量 ===")
daily_stats = (
    log_ts
    .withColumn("date", F.to_date("ts"))
    .groupBy("date")
    .count()
    .orderBy("date")
    .orderBy("date")
)
daily_stats.show(60, truncate=False)
pd_daily_stats = daily_stats.toPandas()
pd_daily_stats.attrs.clear()
pd_daily_stats.to_parquet(os.path.join(local_dir, "10_daily_traffic_stats.parquet"), index=False)


=== 4. 按小时统计请求量（含 date） ===
+----------+----+-----+
|date      |hour|count|
+----------+----+-----+
|2024-01-01|8   |754  |
|2024-01-01|9   |516  |
|2024-01-01|10  |490  |
|2024-01-01|11  |527  |
|2024-01-01|12  |540  |
|2024-01-01|13  |1238 |
|2024-01-01|14  |2034 |
|2024-01-01|15  |2904 |
|2024-01-01|16  |4540 |
|2024-01-01|17  |6312 |
|2024-01-01|18  |7035 |
|2024-01-01|19  |7529 |
|2024-01-01|20  |8049 |
|2024-01-01|21  |7593 |
|2024-01-01|22  |6911 |
|2024-01-01|23  |7232 |
|2024-01-02|0   |7771 |
|2024-01-02|1   |8443 |
|2024-01-02|2   |8009 |
|2024-01-02|3   |6662 |
|2024-01-02|4   |5026 |
|2024-01-02|5   |3751 |
|2024-01-02|6   |2540 |
|2024-01-02|7   |1492 |
|2024-01-02|8   |535  |
|2024-01-02|9   |388  |
|2024-01-02|10  |394  |
|2024-01-02|11  |389  |
|2024-01-02|12  |408  |
|2024-01-02|13  |998  |
|2024-01-02|14  |1588 |
|2024-01-02|15  |2282 |
|2024-01-02|16  |3586 |
|2024-01-02|17  |4841 |
|2024-01-02|18  |5533 |
|2024-01-02|19  |5894 |
|2024-01-02|20  |6106 |
|2024-01-02

### 工作日 vs 周末、高峰时段 vs 低谷的对比汇总

In [ ]:
from pyspark.sql import functions as F

# 已有 log_ts（含 ts 列）时直接使用；否则请先运行你给出的 to_ts_expr 部分构造 log_ts
# 标记工作日/周末 与 高峰时段
log_ts2 = (
    log_ts
    .withColumn("dow", F.dayofweek("ts")) # 1=Sun .. 7=Sat
    .withColumn("is_weekend", F.col("dow").isin(1, 7))
    .withColumn("hour", F.hour("ts"))
)

# 定义高峰时段：示例为早高峰 08-10（含）与 晚高峰 17-20（含）
peak_hours = list(range(8, 11)) + list(range(17, 21))
log_ts2 = log_ts2.withColumn("is_peak", F.col("hour").isin(peak_hours))

# 总体基数（用于计算占比）
total_requests = log_ts2.count()

print(f"\n总请求数: {total_requests}")

# 工作日 vs 周末：按天统计并汇总（总请求数、占比、天数、日均）
wd_vs_we = (
    log_ts2.groupBy("is_weekend")
    .agg(
        F.count("*").alias("total_requests"),
        F.countDistinct("date").alias("distinct_days"),
    )
    .withColumn("pct_of_total", F.round(F.col("total_requests") / F.lit(total_requests) * 100, 2))
    .withColumn("avg_per_day", F.round(F.col("total_requests") / F.col("distinct_days"), 2))
    .orderBy(F.desc("total_requests"))
)
print("\n=== 工作日 vs 周末 汇总 ===")
wd_vs_we.show(truncate=False)
pd_wd_vs_we = wd_vs_we.toPandas()
pd_wd_vs_we.attrs.clear()
pd_wd_vs_we.to_parquet(os.path.join(local_dir, "11_wd_vs_we_traffic_stats.parquet"), index=False)

# 高峰 vs 低谷：总体汇总（总请求数、占比、日均）
peak_vs_off = (
    log_ts2.groupBy("is_peak")
    .agg(
        F.count("*").alias("total_requests"),
        F.countDistinct("date").alias("distinct_days"),
    )
    .withColumn("pct_of_total", F.round(F.col("total_requests") / F.lit(total_requests) * 100, 2))
    .withColumn("avg_per_day", F.round(F.col("total_requests") / F.col("distinct_days"), 2))
    .orderBy(F.desc("total_requests"))
)
print("\n=== 高峰（is_peak=True） vs 低谷（is_peak=False） 汇总 ===")
peak_vs_off.show(truncate=False)
pd_peak_vs_off = peak_vs_off.toPandas()
pd_peak_vs_off.attrs.clear()
pd_peak_vs_off.to_parquet(os.path.join(local_dir, "12_peak_vs_offpeak_traffic_stats.parquet"), index=False)


# 按小时在工作日/周末的分布（用于观察高峰曲线差异）
hourly_by_weektype = (
    log_ts2.groupBy("is_weekend", "hour")
    .count()
    .withColumn("pct_of_total", F.round(F.col("count") / F.lit(total_requests) * 100, 3))
    .orderBy("is_weekend", "hour")
)
print("\n=== 按小时：工作日 vs 周末 ===")
hourly_by_weektype.show(48, truncate=False)
pd_hourly_by_weektype = hourly_by_weektype.toPandas()
pd_hourly_by_weektype.attrs.clear()
pd_hourly_by_weektype.to_parquet(os.path.join(local_dir, "13_hourly_by_weektype_traffic_stats.parquet"), index=False)

# 按小时在高峰/低谷的分布（高分辨率对比）
hourly_by_peak = (
    log_ts2.groupBy("is_peak", "hour")
    .count()
    .withColumn("pct_of_total", F.round(F.col("count") / F.lit(total_requests) * 100, 3))
    .orderBy("is_peak", "hour")
)
print("\n=== 按小时：高峰 vs 低谷 ===")
hourly_by_peak.show(48, truncate=False)
pd_hourly_by_peak = hourly_by_peak.toPandas()
pd_hourly_by_peak.attrs.clear()
pd_hourly_by_peak.to_parquet(os.path.join(local_dir, "14_hourly_by_peak_traffic_stats.parquet"), index=False)

# 把关键汇总合并成单行 pandas 方便保存
pd_summary_extra = {}
for row in wd_vs_we.collect():
    k = "weekend" if row["is_weekend"] else "weekday"
    pd_summary_extra[f"{k}_total_requests"] = row["total_requests"]
    pd_summary_extra[f"{k}_pct_of_total"] = float(row["pct_of_total"])
    pd_summary_extra[f"{k}_avg_per_day"] = float(row["avg_per_day"])

for row in peak_vs_off.collect():
    k = "peak" if row["is_peak"] else "offpeak"
    pd_summary_extra[f"{k}_total_requests"] = row["total_requests"]
    pd_summary_extra[f"{k}_pct_of_total"] = float(row["pct_of_total"])
    pd_summary_extra[f"{k}_avg_per_day"] = float(row["avg_per_day"])

import pandas as _pd
pd_summary_extra = _pd.DataFrame([pd_summary_extra])
pd_summary_extra
pd_summary_extra.to_parquet(os.path.join(local_dir, "15_summary_peak_traffic_stats.parquet"), index=False)



总请求数: 600000

=== 工作日 vs 周末 汇总 ===
+----------+--------------+-------------+------------+-----------+
|is_weekend|total_requests|distinct_days|pct_of_total|avg_per_day|
+----------+--------------+-------------+------------+-----------+
|false     |478693        |9            |79.78       |53188.11   |
|true      |121307        |3            |20.22       |40435.67   |
+----------+--------------+-------------+------------+-----------+


=== 高峰（is_peak=True） vs 低谷（is_peak=False） 汇总 ===
+-------+--------------+-------------+------------+-----------+
|is_peak|total_requests|distinct_days|pct_of_total|avg_per_day|
+-------+--------------+-------------+------------+-----------+
|false  |429911        |10           |71.65       |42991.1    |
|true   |170089        |10           |28.35       |17008.9    |
+-------+--------------+-------------+------------+-----------+


=== 按小时：工作日 vs 周末 ===
+----------+----+-----+------------+
|is_weekend|hour|count|pct_of_total|
+----------+----+-----+------

### 统计URL的错误率

In [36]:
from pyspark.sql import functions as F

# 按小时统计错误率
print("\n=== 5. 按小时统计错误率 ===")
hourly_error = (
    log_ts
    .withColumn("date", F.to_date("ts"))
    .withColumn("hour", F.hour("ts"))
    .groupBy("date", "hour")
    .agg(
        F.count("*").alias("total_requests"),
        F.sum(F.when(F.col("status_code") >= 400, 1).otherwise(0)).alias("error_count")
    )
    .withColumn("error_rate", (F.col("error_count") / F.col("total_requests")) * 100)
    .orderBy("date", "hour")
)
hourly_error.show(200, truncate=False)
pd_hourly_error = hourly_error.toPandas()
pd_hourly_error.attrs.clear()
pd_hourly_error.to_parquet(os.path.join(local_dir, "16_hourly_error_stats.parquet"), index=False)

# 按天统计错误率
print("\n=== 5b. 按日期统计错误率 ===")
daily_error = (
    log_ts
    .withColumn("date", F.to_date("ts"))
    .groupBy("date")
    .agg(
        F.count("*").alias("total_requests"),
        F.sum(F.when(F.col("status_code") >= 400, 1).otherwise(0)).alias("error_count")
    )
    .withColumn("error_rate", (F.col("error_count") / F.col("total_requests")) * 100)
    .orderBy("date")
)
daily_error.show(60, truncate=False)
pd_daily_error = daily_error.toPandas()
pd_daily_error.attrs.clear()
pd_daily_error.to_parquet(os.path.join(local_dir, "17_daily_error_stats.parquet"), index=False)



=== 5. 按小时统计错误率 ===
+----------+----+--------------+-----------+------------------+
|date      |hour|total_requests|error_count|error_rate        |
+----------+----+--------------+-----------+------------------+
|2024-01-01|8   |754           |138        |18.30238726790451 |
|2024-01-01|9   |516           |79         |15.310077519379844|
|2024-01-01|10  |490           |82         |16.73469387755102 |
|2024-01-01|11  |527           |89         |16.888045540796963|
|2024-01-01|12  |540           |82         |15.185185185185185|
|2024-01-01|13  |1238          |222        |17.932148626817447|
|2024-01-01|14  |2034          |363        |17.846607669616517|
|2024-01-01|15  |2904          |492        |16.94214876033058 |
|2024-01-01|16  |4540          |747        |16.45374449339207 |
|2024-01-01|17  |6312          |1060       |16.79340937896071 |
|2024-01-01|18  |7035          |1192       |16.943852167732764|
|2024-01-01|19  |7529          |1210       |16.071191393279317|
|2024-01-01|20  |80

### 慢请求分析（响应时间 > 1000ms）

In [37]:
# 12) 统计分析 6：慢请求分析（响应时间 > 1000ms）
print("\n=== 6. 慢请求分析（>1000ms） ===")
slow_requests = log_ts.filter(F.col("response_time_ms") > 1000)
print("慢请求总数:", slow_requests.count())
slow_requests.groupBy("category","path").count().orderBy(F.col("count").desc()).show(20, truncate=False)
pd_slow_requests = slow_requests.toPandas()
pd_slow_requests.attrs.clear()
pd_slow_requests.to_parquet(os.path.join(local_dir, "18_slow_requests_stats.parquet"), index=False)



=== 6. 慢请求分析（>1000ms） ===
慢请求总数: 108625
+--------+--------------+-----+
|category|path          |count|
+--------+--------------+-----+
|home    |/             |3330 |
|home    |/home         |3326 |
|home    |/index        |3274 |
|home    |/home/index   |3246 |
|home    |/portal       |3202 |
|search  |/search       |2421 |
|login   |/login        |1315 |
|login   |/register     |1287 |
|login   |/auth/register|1269 |
|error   |/not-found    |1243 |
|login   |/signin       |1220 |
|login   |/account/login|1220 |
|error   |/exception    |1212 |
|error   |/500          |1181 |
|error   |/error/404    |1179 |
|error   |/404          |1168 |
|error   |/error/500    |1157 |
|error   |/timeout      |1147 |
|checkout|/checkout/cart|681  |
|checkout|/checkout     |650  |
+--------+--------------+-----+
only showing top 20 rows



### 响应时间的统计

In [38]:
# 16) 统计分析 10：响应时间统计
print("\n=== 10. 响应时间统计 ===")
response_summary = (
    log_ts.agg(
        F.count("*").alias("total_requests"),
        F.avg("response_time_ms").alias("avg_response_time_ms"),
        F.min("response_time_ms").alias("min_response_time_ms"),
        F.max("response_time_ms").alias("max_response_time_ms"),
        F.percentile_approx("response_time_ms", 0.5).alias("p50_response_time_ms"),
        F.percentile_approx("response_time_ms", 0.95).alias("p95_response_time_ms")
    )
)
response_summary.show(truncate=False)
pd_response_summary = response_summary.toPandas()
pd_response_summary.attrs.clear()
pd_response_summary.to_parquet(os.path.join(local_dir, "19_response_summary_stats.parquet"), index=False)


# 17) 统计分析 11：按请求路径 Top 20
print("\n=== 11. Top 20 请求路径 ===")
path_stats = (
    log_ts.groupBy("path")
    .count()
    .orderBy(F.col("count").desc())
    .limit(20)
)
path_stats.show(20, truncate=False)
pd_path_stats = path_stats.toPandas()
pd_path_stats.attrs.clear()
pd_path_stats.to_parquet(os.path.join(local_dir, "20_response_path_stats.parquet"), index=False)


=== 10. 响应时间统计 ===
+--------------+--------------------+--------------------+--------------------+--------------------+--------------------+
|total_requests|avg_response_time_ms|min_response_time_ms|max_response_time_ms|p50_response_time_ms|p95_response_time_ms|
+--------------+--------------------+--------------------+--------------------+--------------------+--------------------+
|600000        |812.6388666666667   |20                  |14999               |383                 |3689                |
+--------------+--------------------+--------------------+--------------------+--------------------+--------------------+


=== 11. Top 20 请求路径 ===
+---------------+-----+
|path           |count|
+---------------+-----+
|/              |19762|
|/index         |19659|
|/home/index    |19524|
|/home          |19501|
|/portal        |19428|
|/search        |14131|
|/auth/register |7743 |
|/login         |7704 |
|/register      |7621 |
|/signin        |7563 |
|/account/login |7499 |
|/checko

### 按 method、status、country、category 统计响应时间

In [39]:
# 按 method 汇总响应时间
print("\n=== 响应时间汇总：按 method ===")
if "method" in log_ts.columns:
    resp_by_method = (
        log_ts.groupBy("method")
        .agg(
            F.count("*").alias("count"),
            F.avg("response_time_ms").alias("avg_ms"),
            F.expr("percentile_approx(response_time_ms, 0.5)").alias("p50_ms"),
            F.expr("percentile_approx(response_time_ms, 0.95)").alias("p95_ms"),
            F.min("response_time_ms").alias("min_ms"),
            F.max("response_time_ms").alias("max_ms"),
        )
        .orderBy(F.desc("count"))
    )
    resp_by_method.show(200, truncate=False)
    pd_resp_by_method = resp_by_method.toPandas()
    pd_resp_by_method.attrs.clear()
    pd_resp_by_method.to_parquet(os.path.join(local_dir, "21_response_by_method_stats.parquet"), index=False)
else:
    print("method 列不存在，跳过按 method 汇总")

# 按 status 大类（2xx/3xx/4xx/5xx/other）汇总响应时间
print("\n=== 响应时间汇总：按 status 大类 ===")
resp_with_class = log_ts.withColumn(
    "status_class",
    F.when((F.col("status_code") >= 200) & (F.col("status_code") < 300), "2xx")
     .when((F.col("status_code") >= 300) & (F.col("status_code") < 400), "3xx")
     .when((F.col("status_code") >= 400) & (F.col("status_code") < 500), "4xx")
     .when((F.col("status_code") >= 500) & (F.col("status_code") < 600), "5xx")
     .otherwise("other")
)

resp_by_status_class = (
    resp_with_class.groupBy("status_class")
    .agg(
        F.count("*").alias("count"),
        F.avg("response_time_ms").alias("avg_ms"),
        F.expr("percentile_approx(response_time_ms, 0.5)").alias("p50_ms"),
        F.expr("percentile_approx(response_time_ms, 0.95)").alias("p95_ms"),
        F.min("response_time_ms").alias("min_ms"),
        F.max("response_time_ms").alias("max_ms"),
    )
    .orderBy(F.desc("count"))
)
resp_by_status_class.show(20, truncate=False)
pd_resp_by_status_class = resp_by_status_class.toPandas()
pd_resp_by_status_class.attrs.clear()
pd_resp_by_status_class.to_parquet(os.path.join(local_dir, "22_response_by_status_class_stats.parquet"), index=False)


# 按 country 汇总响应时间（Top 50 国家）
print("\n=== 响应时间汇总：按 country（Top 50） ===")
if "country" in log_ts.columns:
    resp_by_country = (
        log_ts.groupBy("country", "country_name")
        .agg(
            F.count("*").alias("count"),
            F.avg("response_time_ms").alias("avg_ms"),
            F.expr("percentile_approx(response_time_ms, 0.5)").alias("p50_ms"),
            F.expr("percentile_approx(response_time_ms, 0.95)").alias("p95_ms"),
            F.min("response_time_ms").alias("min_ms"),
            F.max("response_time_ms").alias("max_ms"),
        )
        .orderBy(F.desc("count"))
        .limit(50)
    )
    resp_by_country.show(50, truncate=False)
    pd_resp_by_country = resp_by_country.toPandas()
    pd_resp_by_country.attrs.clear()
    pd_resp_by_country.to_parquet(os.path.join(local_dir, "23_response_by_country_stats.parquet"), index=False)
else:
    print("country 列不存在，跳过按 country 汇总")

# 按 category 汇总响应时间（Top 50 category）
print("\n=== 响应时间汇总：按 category（Top 50） ===")
if "category" in log_ts.columns:
    resp_by_category = (
        log_ts.groupBy("category")
        .agg(
            F.count("*").alias("count"),
            F.avg("response_time_ms").alias("avg_ms"),
            F.expr("percentile_approx(response_time_ms, 0.5)").alias("p50_ms"),
            F.expr("percentile_approx(response_time_ms, 0.95)").alias("p95_ms"),
            F.min("response_time_ms").alias("min_ms"),
            F.max("response_time_ms").alias("max_ms"),
        )
        .orderBy(F.desc("count"))
        .limit(50)
    )
    resp_by_category.show(50, truncate=False)
    pd_resp_by_category = resp_by_category.toPandas()
    pd_resp_by_category.attrs.clear()
    pd_resp_by_category.to_parquet(os.path.join(local_dir, "24_response_by_category_stats.parquet"), index=False)
else:
    print("category 列不存在，跳过按 category 汇总")




=== 响应时间汇总：按 method ===
+------+------+-----------------+------+------+------+------+
|method|count |avg_ms           |p50_ms|p95_ms|min_ms|max_ms|
+------+------+-----------------+------+------+------+------+
|GET   |148683|810.0470127721394|381   |3699  |20    |14999 |
|POST  |109286|834.8631297695954|383   |3738  |20    |14990 |
|PUT   |93942 |836.5083349300633|386   |3732  |20    |14998 |
|DELETE|93297 |777.6424322325477|379   |3612  |20    |14999 |
|HEAD  |77899 |846.5889806030887|388   |3766  |20    |14977 |
|PATCH |76893 |764.9700362841871|382   |3562  |20    |5000  |
+------+------+-----------------+------+------+------+------+


=== 响应时间汇总：按 status 大类 ===
+------------+------+------------------+------+------+------+------+
|status_class|count |avg_ms            |p50_ms|p95_ms|min_ms|max_ms|
+------------+------+------------------+------+------+------+------+
|2xx         |416614|629.3905413644285 |351   |3289  |20    |6997  |
|3xx         |83896 |637.0310026699724 |354   |331

In [40]:
# Window 示例：每个 country 的 Top 3 paths（按访问量）
from pyspark.sql.window import Window
w = Window.partitionBy("country").orderBy(F.desc("cnt"))
path_cnt = log_ts.groupBy("country","path").count().withColumnRenamed("count","cnt")
top3 = path_cnt.withColumn("rank", F.row_number().over(w)).filter(F.col("rank") <= 3)
print("\n=== Top 3 paths per country (Window) ===")
top3.show(50, truncate=False)
pd_top3 = top3.toPandas()
pd_top3.attrs.clear()
pd_top3.to_parquet(os.path.join(local_dir, "25_top3_paths_per_country.parquet"), index=False)


=== Top 3 paths per country (Window) ===
+-------+-----------+----+----+
|country|path       |cnt |rank|
+-------+-----------+----+----+
|UA     |/home/index|294 |1   |
|UA     |/index     |290 |2   |
|UA     |/          |281 |3   |
|NL     |/index     |311 |1   |
|NL     |/home/index|305 |2   |
|NL     |/          |300 |3   |
|MX     |/home/index|321 |1   |
|MX     |/portal    |300 |2   |
|MX     |/          |297 |3   |
|CN     |/index     |3574|1   |
|CN     |/          |3485|2   |
|CN     |/home/index|3448|3   |
|RU     |/portal    |300 |1   |
|RU     |/home      |300 |2   |
|RU     |/home/index|295 |3   |
|ID     |/          |307 |1   |
|ID     |/index     |287 |2   |
|ID     |/home      |285 |3   |
|AU     |/          |606 |1   |
|AU     |/index     |583 |2   |
|AU     |/home/index|580 |3   |
|SA     |/portal    |296 |1   |
|SA     |/index     |293 |2   |
|SA     |/          |292 |3   |
|CA     |/home/index|598 |1   |
|CA     |/home      |595 |2   |
|CA     |/index     |580 |3   

In [24]:
# 性能对比：不同 persist/cache 策略
from time import perf_counter
perf_records = []

def measure_count(df, label, do_persist=None):
    if do_persist is not None:
        df.persist(do_persist)
    t0 = perf_counter()
    cnt = df.count()
    t1 = perf_counter()
    if do_persist is not None:
        df.unpersist()
    perf_records.append({"strategy": label, "time_s": t1-t0, "count": cnt})
    print(f"{label}: count={cnt}, time={t1-t0:.3f}s")

# baseline on log_geo_url_df (already persisted earlier)
measure_count(log_ts, "persisted(MEM+DISK) existing")

# unpersist then measure no cache
log_ts.unpersist()
measure_count(log_ts, "no_cache")

# cache MEMORY_ONLY
log_ts.cache()
measure_count(log_ts, "cache(MEMORY_ONLY)")
log_ts.unpersist()

# persist MEMORY_ONLY
from pyspark import StorageLevel
measure_count(log_ts.persist(StorageLevel.MEMORY_ONLY), "persist(MEMORY_ONLY)")
log_ts.unpersist()

# persist MEMORY_AND_DISK
measure_count(log_ts.persist(StorageLevel.MEMORY_AND_DISK), "persist(MEMORY_AND_DISK)")
log_ts.unpersist()

# # 将 perf_records 写入本地 parquet 以便后续分析
# import os
# local_dir = r"G:\Log-Spark-Insight-Analysis-v2\output_further_analytics"
# os.makedirs(local_dir, exist_ok=True)
# spark.createDataFrame(perf_records).toPandas().to_parquet(os.path.join(local_dir, "perf_comparison.parquet"), index=False)


persisted(MEM+DISK) existing: count=600000, time=0.380s
no_cache: count=600000, time=0.370s
cache(MEMORY_ONLY): count=600000, time=3.019s
persist(MEMORY_ONLY): count=600000, time=2.604s
persist(MEMORY_AND_DISK): count=600000, time=2.667s


DataFrame[log_id: string, timestamp: bigint, date: string, hour: bigint, ip: string, method: string, url: string, path: string, status_code: bigint, response_time_ms: bigint, bytes: bigint, user_agent: string, country: string, referer: string, is_anomaly: bigint, anomaly_type: string, country_code: string, country_name: string, region: string, continent: string, is_high_risk: boolean, path_pattern: string, category: string, is_api: boolean, is_static: boolean, priority: bigint, ts: timestamp]

In [ ]:
# 18) 保存分析结果到 HDFS
# out_base = f"{hdfs_base}/spark/httplog_analytics/results"
# country_stats.write.mode("overwrite").parquet(f"{out_base}/country_stats")
# category_stats.write.mode("overwrite").parquet(f"{out_base}/category_stats")
# status_stats.write.mode("overwrite").parquet(f"{out_base}/status_stats")
# hourly_stats.write.mode("overwrite").parquet(f"{out_base}/hourly_stats")
# daily_error.write.mode("overwrite").parquet(f"{out_base}/daily_error")
# anomaly_ip_stats.write.mode("overwrite").parquet(f"{out_base}/anomaly_ip_stats")
# path_stats.write.mode("overwrite").parquet(f"{out_base}/path_stats")
# response_summary.write.mode("overwrite").parquet(f"{out_base}/response_summary")
# high_risk_country.write.mode("overwrite").parquet(f"{out_base}/high_risk_country")


# print("\n=== 分析结果已写入 HDFS ===")
# print(f"结果目录：{out_base}")


In [32]:
# 19) 关闭 Spark
spark.stop()